# Task 1 — GPO Annual Report: Baseline RAG

**Document:** `GPO-Anual-report.pdf` (GPO 2018 Annual Report, 82 pages, mostly narrative text)
**Goal:** a solid, *measured* baseline RAG pipeline: ingestion → chunking → embeddings → retrieval → answer generation.
**Pass bar (task spec):** the exact fact from the surrounding paragraph, with nothing added that isn't in the document.

**How it's evaluated (nothing document-specific, nothing tuned to pass):**
- **Real pipeline.** Every answer comes from the app's real `RAGPipeline.ask()`: the same parsing, chunking,
  embeddings, retrieval, query rewriting and strict mode as the chat. Only the knob under test changes.
- **Ground truth only.** The questions file (`evaluation/questions/gpo.json`) holds the question, a plain-language
  reference answer, and the pages that state it. Expected pages were located by searching the PDF text.
- **AI judge, no keyword lists.** Answers are graded by a pinned model at temperature 0 against the reference
  answer. Section 6 checks that the judge actually rejects wrong answers.
- **One general harness.** `evaluation/harness.py` and the CLI `python -m evaluation.run --questions <file>` work
  for any document and questions file.

**Questions:**
- The 3 sample questions from the task spec.
- 9 extra fact questions.
- 1 trap question whose answer is *not* in the report; the system must decline instead of guessing.

In [1]:
import sys, time, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # project root: app/ and evaluation/
warnings.filterwarnings("ignore")
import logging; logging.disable(logging.WARNING)

import pandas as pd
from pypdf import PdfReader
from app.config.settings import settings
from evaluation.harness import (EvalConfig, build_pipeline, evaluate_answers, evaluate_retrieval, grade_answer,
                                load_questions, set_retrieval, summarize_answers, summarize_retrieval)

pd.set_option("display.max_colwidth", 140)
QS = load_questions("evaluation/questions/gpo.json")
DOC, QUESTIONS = QS["document"], QS["questions"]
pd.DataFrame([{"id": q["id"], "question": q["question"], "reference answer": q["reference_answer"],
               "pages": q["expected_pages"]} for q in QUESTIONS])

,id,question,reference answer,pages
0,sample-1,How many depository libraries did GPO staff visit in FY 2018?,GPO staff visited 166 depository libraries (in 25 states) in FY 2018.,"[5, 21]"
1,sample-2,"What new digital system replaced FDsys, and when was it fully deployed?","govinfo replaced FDsys. Its full-scale deployment happened over FY 2018: it moved out of beta in December 2017, ran in parallel with FDs...","[4, 8, 10, 22, 39]"
2,sample-3,How much did GPO procure in printing and information products in FY 2018 on behalf of federal agencies?,"About $375.7 million (the report also says 'more than $375 million') of printing and other information products, across more than 84,000...","[5, 14, 15, 17]"
3,copies-distributed,How many copies and titles did GPO distribute to depository libraries in FY 2018?,"802,834 copies of 4,211 titles.","[5, 15, 19]"
4,passports,How many passports did GPO produce in FY 2018?,"15,901,600 passports (the narrative sections round this to about 16 million).","[6, 14, 15, 18]"
5,fdlp-webinars,"How many FDLP Academy webinars were hosted in FY 2018, and how many participants did they serve?","Over 85 FDLP Academy webinars, serving more than 10,000 participants.",[5]
6,revenues,What were GPO's total revenues in FY 2018?,"$874,466,000 (also stated as $874.5 million).","[6, 14, 15, 25, 26, 43, 59]"
7,staff,How many staff did GPO have in FY 2018?,"1,737 staff.","[14, 15]"
8,libraries-nationwide,How many depository libraries are there nationwide?,"1,133 depository libraries nationwide.","[8, 15, 19, 39]"
9,nara-renovation,How much space did GPO renovate in Building A for NARA's Center for Legislative Archives?,"Over 24,000 square feet (another section says approximately 25,000 square feet).","[5, 17]"


## 1. Ingestion check

What does text extraction give us? Pages with almost no text (covers, photo pages, infographics) are blind spots
for a text-only pipeline.

In [2]:
reader = PdfReader(Path.cwd().parent / DOC)
chars = [len((p.extract_text() or "").strip()) for p in reader.pages]
low = [i for i, c in enumerate(chars, 1) if c < 200]
print(f"{len(chars)} pages | avg {sum(chars)//len(chars)} chars/page | low-text pages (<200 chars): {low}")

82 pages | avg 2169 chars/page | low-text pages (<200 chars): [1, 2, 29, 30, 31, 32, 50, 51, 53, 57]


## 2. Experiment — chunk size × top-k (embedding model fixed: `all-MiniLM-L6-v2`)

Retrieval only, no LLM calls.
- **hit rate:** share of questions where a chunk from an expected page came back.
- **MRR:** mean reciprocal rank of the first correct chunk (1.0 = always ranked first).

Threshold is the app default, 0.2.

In [3]:
pipelines, rows = {}, []
for size, overlap in [(500, 100), (800, 150), (1200, 200)]:
    t = time.time()
    pipe, n_chunks = build_pipeline(DOC, EvalConfig(chunk_size=size, chunk_overlap=overlap))
    pipelines[size] = pipe
    for k in (3, 5, 8):
        set_retrieval(pipe, top_k=k, score_threshold=0.2)
        r = evaluate_retrieval(pipe, QUESTIONS)
        rows.append({"chunking": f"{size}/{overlap}", "chunks": n_chunks, "build_s": round(time.time() - t),
                     "top_k": k, **summarize_retrieval(r), "missed": [x["id"] for x in r if not x["hit"]]})
pd.DataFrame(rows)

,chunking,chunks,build_s,top_k,hit_rate,mrr,avg_chunks,missed
0,500/100,478,38,3,0.917,0.875,3.0,[preservation-stewards]
1,500/100,478,38,5,0.917,0.875,5.0,[preservation-stewards]
2,500/100,478,39,8,0.917,0.875,8.0,[preservation-stewards]
3,800/150,297,23,3,0.917,0.819,3.0,[preservation-stewards]
4,800/150,297,24,5,1.000,0.836,5.0,[]
5,800/150,297,25,8,1.000,0.836,8.0,[]
6,1200/200,209,14,3,0.917,0.764,3.0,[preservation-stewards]
7,1200/200,209,15,5,0.917,0.764,5.0,[preservation-stewards]
8,1200/200,209,16,8,0.917,0.764,8.0,[preservation-stewards]


## 3. Experiment — embedding model

The current local model vs a stronger small retrieval model, `BAAI/bge-small-en-v1.5`, at the default chunking.
BGE expects an instruction on queries, which the app now supports via `EMBEDDING_QUERY_PREFIX`.

In [4]:
BGE = EvalConfig(embedding_model="BAAI/bge-small-en-v1.5",
                 embedding_query_prefix="Represent this sentence for searching relevant passages: ")
t = time.time(); bge, _ = build_pipeline(DOC, BGE); bge_build = round(time.time() - t)
rows = []
for name, pipe, build_s in [("all-MiniLM-L6-v2", pipelines[800], None), ("bge-small-en-v1.5", bge, bge_build)]:
    for k in (5, 8):
        set_retrieval(pipe, top_k=k, score_threshold=0.2)
        r = evaluate_retrieval(pipe, QUESTIONS)
        rows.append({"model": name, "build_s": build_s, "top_k": k, **summarize_retrieval(r),
                     "missed": [x["id"] for x in r if not x["hit"]]})
pd.DataFrame(rows)

,model,build_s,top_k,hit_rate,mrr,avg_chunks,missed
0,all-MiniLM-L6-v2,NaN,5,1.0,0.836,5.0,[]
1,all-MiniLM-L6-v2,NaN,8,1.0,0.836,8.0,[]
2,bge-small-en-v1.5,68.0,5,1.0,0.917,5.0,[]
3,bge-small-en-v1.5,68.0,8,1.0,0.917,8.0,[]


## 4. Experiment — similarity threshold (MiniLM, 800/150, top-k 5)

In [5]:
base = pipelines[800]
rows = []
for th in (0.0, 0.1, 0.2, 0.3, 0.4):
    set_retrieval(base, top_k=5, score_threshold=th)
    r = evaluate_retrieval(base, QUESTIONS)
    rows.append({"threshold": th, **summarize_retrieval(r), "missed": [x["id"] for x in r if not x["hit"]]})
set_retrieval(base, top_k=settings.top_k, score_threshold=settings.similarity_threshold)  # back to app defaults
pd.DataFrame(rows)

,threshold,hit_rate,mrr,avg_chunks,missed
0,0.0,1.0,0.836,5.0,[]
1,0.1,1.0,0.836,5.0,[]
2,0.2,1.0,0.836,5.0,[]
3,0.3,1.0,0.836,5.0,[]
4,0.4,1.0,0.836,4.8,[]


## 5. End-to-end answers with the app's default configuration

Strict document mode, query rewriting on, exactly as in the chat. The answer model is **pinned** (configured Gemini
model, temperature 0), because the app default `openrouter/auto` routes each call to a different model, so runs
wouldn't be comparable. Calls are spaced to 12/min: Gemini's free tier allows 15/min, and going over silently falls
back to another provider.

In [6]:
MODEL = settings.gemini_model
print("Config:", EvalConfig().label, "| answer + judge model:", MODEL)
answers = evaluate_answers(base, QUESTIONS, model_name=MODEL, calls_per_minute=12)
print(summarize_answers(answers))
pd.DataFrame([{"id": r["id"], "correct": r["correct"], "cites page": r["citation_ok"],
               "pages retrieved": sorted({p for p in r["retrieved_pages"] if p}),
               "answer": r["answer"].replace("\n", " ")[:260], "judge": r["judge_reason"]} for r in answers])

Config: all-MiniLM-L6-v2 | 800/150 | k=5 | th=0.2 | answer + judge model: gemini-3.5-flash-lite


{'accuracy': 1.0, 'citation_accuracy': 1.0}


,id,correct,cites page,pages retrieved,answer,judge
0,sample-1,True,True,"[5, 21, 73]","In FY 2018, GPO staff visited 166 depository libraries in 25 states (Source: `GPO-Anual-report.pdf`, pages 5 and 21).","The system answer accurately states the number of depository libraries visited in FY 2018 along with the number of states, which matches..."
1,sample-2,True,True,"[8, 10, 39, 46]","The digital system that replaced FDsys is **govinfo** (Source: GPO-Anual-report.pdf, pages 8, 10, and 39). Regarding when it was fully...",The system correctly identified govinfo as the replacement system and provided the December 2018 full deployment date as stated in the r...
2,sample-3,True,True,"[5, 14, 17, 62]","In FY 2018, work done through GPO offices (procured printing/jobs procured nationwide) was valued at approximately $375.7 million, repre...","The system answer accurately provides the key figure of approximately $375.7 million and the order count from FY 2018, matching the refe..."
3,copies-distributed,True,True,"[5, 15, 19, 21]","In FY 2018, GPO distributed 802,834 copies of 4,211 titles to depository libraries nationwide (Source: GPO-Anual-report.pdf, pages 5 and...",The system answer accurately provides the exact number of copies and titles mentioned in the reference answer.
4,passports,True,True,"[6, 14, 18, 26, 49]","In FY 2018, GPO produced approximately 16 million passports for the State Department (specifically, 15,901,600 passports according to th...","The system answer includes both the exact figure of 15,901,600 and the rounded approximate figure of 16 million, matching the reference ..."
5,fdlp-webinars,True,True,"[4, 5, 6, 20, 21]","In FY 2018, the FDLP Academy hosted over 85 webinars (specifically noting 85 webinars with 10,214 combined registrants) that served more...",The system answer accurately captures the key facts from the reference answer regarding the number of webinars and participants in FY 20...
6,revenues,True,True,"[14, 25, 26, 27]","GPO's total revenues in FY 2018 were $874.5 million (or $874,466,000) (Source: GPO-Anual-report.pdf, pages 14, 25, and 26).",The system accurately provides GPO's total revenues for FY 2018 in both exact and rounded formats matching the reference answer.
7,staff,True,True,"[4, 12, 14, 26]","In FY 2018, GPO had 1,737 employees on board (Source: GPO-Anual-report.pdf, pages 12 and 14).","The system answer correctly identifies the number of staff as 1,737 in FY 2018, matching the reference answer."
8,libraries-nationwide,True,True,"[5, 15, 19]","There are 1,133 depository libraries nationwide (Source: GPO-Anual-report.pdf, pages 15 and 19).","The system answer accurately provides the correct number of depository libraries nationwide as stated in the reference answer, with acce..."
9,nara-renovation,True,True,"[5, 17, 77]","GPO renovated approximately 25,000 square feet of space (also referred to in another section as over 24,000 square feet) in Building A t...",The system answer accurately captures the renovated space figures provided in the reference answer.


## 6. Does the judge actually catch wrong answers?

A judge that passes everything proves nothing. Feed it deliberately bad answers. It must fail:
- a wrong number;
- a hedge;
- a real figure from the wrong context ($341.5M is printing *expenses* on p. 25, not procurement);
- a made-up answer to the trap question.

It must still pass a validly rounded figure.

In [7]:
by_id = {q["id"]: q for q in QUESTIONS}
cases = [
    ("sample-1", "GPO staff visited 116 depository libraries in FY 2018 (p. 5).", "wrong number", False),
    ("sample-2", "govinfo replaced FDsys, but the report does not specify when it was fully deployed.", "hedge", False),
    ("sample-3", "GPO procured about $341.5 million of printing in FY 2018 (p. 25).", "nearby wrong figure", False),
    ("trap-not-in-report", "GPO's FY 2025 budget was $117 million.", "invented trap answer", False),
    ("passports", "GPO produced approximately 16 million passports in FY 2018 (p. 18).", "valid rounding", True),
]
rows = []
for qid, answer, kind, should_pass in cases:
    time.sleep(5)  # stay under the free-tier rate limit
    v = grade_answer(base.generator, by_id[qid]["question"], by_id[qid]["reference_answer"], answer, MODEL)
    rows.append({"case": kind, "expected": "PASS" if should_pass else "FAIL",
                 "judge": "PASS" if v["correct"] else "FAIL", "agrees": v["correct"] == should_pass,
                 "reason": v["reason"]})
calibration = pd.DataFrame(rows)
print("Judge agreement:", f"{calibration['agrees'].mean():.0%}")
calibration

Judge agreement: 100%


,case,expected,judge,agrees,reason
0,wrong number,FAIL,FAIL,True,"The system answer states 116 depository libraries, whereas the reference answer states 166."
1,hedge,FAIL,FAIL,True,"The system answer incorrectly states that the report does not specify when govinfo was fully deployed, whereas the reference answer prov..."
2,nearby wrong figure,FAIL,FAIL,True,"The system answer states $341.5 million, which contradicts the reference answer of about $375.7 million."
3,invented trap answer,FAIL,FAIL,True,"The system provided a budget figure, whereas the reference states that the document does not contain this information."
4,valid rounding,PASS,PASS,True,"The system answer correctly identifies that GPO produced approximately 16 million passports in FY 2018, which aligns with the rounded fi..."


## 7. Conclusions

**Task 1 passes.** With ALLY AI's default configuration, the real pipeline answers **13/13** questions correctly
(3 sample + 9 extra + 1 trap), cites a correct page **every time**, and declines the question whose answer isn't
in the report. The AI judge agreed with the expected verdict on **5/5** deliberately bad or good answers, so the
score isn't a lenient grader.

**Chosen baseline configuration** (all are the app's existing defaults; nothing was changed to pass):

| Stage | Choice | Evidence |
|---|---|---|
| Ingestion | `pypdf` + text cleaning | Narrative PDF extracts cleanly; the 10 low-text pages are covers and photos |
| Chunking | 800 chars, 150 overlap (recursive, paragraph/sentence boundaries first) | Only size with a 100% hit rate; 500 and 1200 both miss a short fact |
| Embeddings | `all-MiniLM-L6-v2`, local | 100% hit rate. `bge-small` ranks better (MRR 0.92 vs 0.84) but indexes ~3× slower; revisit for task 2 |
| Retrieval | top-5, similarity ≥ 0.2, LLM query rewriting | top-8 adds nothing; threshold insensitive from 0.0 to 0.4 |
| Generation | Strict document mode, page citations | No invented facts; declines out-of-scope questions |

**Lessons that carry into tasks 2 and 3:**
1. **Ground truth must cover every way the document states a fact.** This report gives some figures in several
   places with different precision: revenue as $874,466,000 and $874.5 million (7 pages); passports as 15,901,600
   and "16 million"; NARA space as "over 24,000" and "approximately 25,000" sq ft. Reference answers mention every
   variant, and expected pages list every page that states the fact.
2. **Grade meaning, not keywords.** A keyword checker first marked correct answers wrong and needed per-question
   tuning, so it was replaced by an LLM judge against the reference answer, verified in section 6.
3. **Pin the LLM and respect rate limits.** `openrouter/auto` picks a different model per call. Gemini's free tier
   (15/min) silently falls back to another provider when exceeded. Both break comparability; the harness pins the
   model and throttles calls.

**Baseline to beat in tasks 2 and 3:** hit rate 1.00 · MRR 0.84 · answer accuracy 1.00 · citation accuracy 1.00.

Re-run any time, for this or any other document: `python -m evaluation.run --questions evaluation/questions/gpo.json`